## 1.1 Testing transforms and plotting 

In [ ]:
import sys
sys.path.append('..')

import matplotlib.pyplot as plt
from scipy import signal

import pywt
import numpy as np
import random

from wavelet_transform.test import synthetic_data as sd
from wavelet_transform.wpt import transform as wpt
from wavelet_transform.postprocess import plotting as plot
from wavelet_transform.utils.extensions import sci_form
from wavelet_transform.test import plotting as test_plot

In [ ]:
amp_sin_waves = 1 # amplitude of the sine waves added together to form the signal
sampling_rate = 50e6
phase = 0

frequency = 2e6

frequencies = [frequency] # create list of frequencies for synthetic data
time_period = 1/frequency
signal_duration = 100*time_period # duration of the signal is 10 times the test frequency
delay = 0#2*time_period # signal starts 2 time periods late
end = signal_duration#8*time_period # signal ends 2 time periods early

gaussian_noise_std = 0.01 # set the standard devaition of the noise

# generate data
data, t_plot = sd.sound_wave(
    amp_sin_waves, frequencies, phase, signal_duration, 
    sampling_rate, delay, end
    )

# add noise
gaussian_noise = sd.noise(gaussian_noise_std,signal_duration,sampling_rate)
data += gaussian_noise
data = data[:2**10]
t_plot = t_plot[:2**10]
data = signal.windows.hann(len(data))*data

# transform parameters
levels = 3
db_order = 1
wavelet = f'db{db_order}'
mode = "zero"

# transform
transform_object = wpt.dwpt(data,levels, wavelet, mode)
final_level_nodes = transform_object.get_level(levels)
dwt_nodes = transform_object.dwt_results() # the dwt results can be extracted from the dwpt

# calculate energy
transform_object.calc_frequencies(sampling_rate)

# set title
f = sci_form(frequency)
title = f"- signal frequency: \n{f} Hz"

# plot everything
plot.full_plot(
    final_level_nodes, data, t_plot, sampling_rate, 
    title, frequencies
    )

### Total Energy
Calculating total energy at different levels

In [ ]:
from wavelet_transform.utils.statistical_features import energy
t = np.linspace(0, 10, 1000)
data = np.sin(2 * np.pi * t)
data += np.random.normal(0, 1, len(t))
levels = 5
wavelet = "haar"
mode = "zero"
transform_object = wpt.dwpt(data, levels, wavelet, mode)

info = transform_object.info(level=levels, order="freq")
# total_energy = sum(info["energy"])
total_energy = sum(node.energy for node in transform_object.get_level())
total_energy_data = energy(data)
print("sum of node energy:", total_energy)
print("signal energy:", total_energy_data)
print("energy of each node:")
display(info["energy"])

total_energy_dwt = sum(node.energy for node in transform_object.dwt_results())
print("total dwt energy:", total_energy_dwt)

## 1.2 Testing with pywt library

In [ ]:
# Test result against pywt library

# Generate data
data = [random.random() for _ in range(50000)]

# Transform settings
levels = 5
wavelet = "haar"
mode = "zero"
pywp_object = pywt.WaveletPacket(data, wavelet, maxlevel=levels, mode=mode)

# pywt result
print("pywt result")
pywt_result = test_plot.plot_pywp_test(pywp_object, levels)

# Library result
print("library result")
obj = wpt.dwpt(data, levels, wavelet, mode)
library_result = test_plot.plot_test(obj.get_level(levels))

# Error calculation
error = pywt_result - library_result
print("total error", np.sum(error))
plt.plot(error)
plt.title("Error")
plt.show()

## 1.3 specifying nodes to plot

In [ ]:
# find nodes with frequencies of interest
frequencies_of_interest = [1e6,1e7,0.5e7]

nodes_of_interest = transform_object.find_nodes(
    final_level_nodes,
    frequencies=frequencies_of_interest
)

# plot data from selected nodes
rows = int(len(nodes_of_interest))

fig,ax = plt.subplots(rows)
plot.plot_nodes(nodes_of_interest,ax,sampling_rate,rows,columns = 1)
fig.set_size_inches(12,3*rows)
fig.tight_layout()

## 2.1 Testing inverse transform

In [ ]:
sys.path.append('..')

from wavelet_transform.wpt import inverse_transform as inv
from wavelet_transform.wpt import transform as wpt
import numpy as np
from wavelet_transform.test import synthetic_data as sd

from scipy.signal import blackmanharris
import matplotlib.pyplot as plt

In [ ]:
# transform
levels = 8
wavelet = 'db38'
sf = 50e6
freq = 2e6
amp_sin_waves = 1
phase = 0
signal_duration = 1000/sf
data, t_plot = sd.sound_wave(
    amp_sin_waves, frequency, phase, signal_duration, 
    sampling_rate=sf
    )
data = data * blackmanharris(len(data))
plt.plot(data)
plt.title("data")
plt.show()

# Wavelet transform of the data
transform_object = wpt.dwpt(data, levels, wavelet)
transform_object.calc_frequencies(sampling_rate)

# inverse dwt
dwt_inverse = inv.inv_dwt(transform_object)
print('total error dwt:', sum(data - dwt_inverse))

# inverse dwpt
dwpt_inverse = inv.inv_dwpt(transform_object)
print('total error, packet', sum(np.array(data) - dwpt_inverse))
plt.plot(dwpt_inverse)
plt.title("recovered signal")
plt.show()

In [ ]:
# testing the reconstruction of separate nodes
rec_1 = inv.inv_dwpt(transform_object, nodes=50)
rec_0 = inv.inv_dwpt(transform_object, nodes=0)
rec_1_0 = inv.inv_dwpt(transform_object, nodes=[0, 50])
error = rec_1_0 - rec_1 - rec_0
print("average error", np.mean(error))

### Info Attributes
Info attributes display statistical measures at each node. InverseTransform.info() calculates these on the recovered signal at each node, while the Transform.info() function calculates these on the wavelet coefficinets at each node.

In [ ]:
inv_transform_obj = inv.InverseTransform(transform_object)
inv_transform_obj.info(level=levels, order="freq")

In [ ]:
transform_object.info(level=levels)

In [ ]:
# Inverse transform from a single node
rec_0 = inv.inv_dwpt(transform_object, frequencies=1)
plt.plot(t_plot, rec_0)
plt.title("Signal recovered from node containing frequency: 1hz")
plt.show()

## 2.2 Testing inverse transform with synthetic data

The data is generated using the synthetic data package. Nodes are chosen for reconstruction and then the reconstructed sine wave is plotted alongside the transform results.

In [ ]:
from numpy.random import seed

seed(seed = 1)

# set parameters for synthetic data
amp_sin_waves = 1 # amplitude of each sine wave that is added to the sound wave
sampling_rate = 30e6

test_frequencies = [1e6] # list of frequencies to test
harmonics = [1/4, 1/2, 1, 2, 3] # list of harmonics to test for each frequency

for frequency in test_frequencies:
    frequencies = frequency*np.array(harmonics)

    time_period = 1/frequency
    signal_duration = 10*time_period # signal duration is 10 times the period of the central frequency
    delay = 2*time_period # signal starts 2 time periods late
    end = 8*time_period # signal ends 2 time periods early

    gaussian_noise_std = 5 # set the standard devaition of the noise

    # generate data
    data, t_plot = sd.sound_wave(
        amp_sin_waves, frequencies, phase, signal_duration, 
        sampling_rate, delay, end
        )

    gaussian_noise = sd.noise(gaussian_noise_std, signal_duration, sampling_rate)
    data += gaussian_noise
    data = signal.windows.hann(len(data)) * data

    # set parameters for transform
    levels = 2
    db_order = 9
    wavelet = f'db{db_order}'

    # transform
    transform_object = wpt.dwpt(data, levels, wavelet)
    final_nodes = transform_object.get_level(levels) # collect nodes from the last level
    transform_object.calc_frequencies(sampling_rate) # calculate the frequency ranges for each node

    nodes = ["aa", "ad"]  
    # takes same keyword arguments as packet_nodes_for_rec
    inverse = inv.inv_dwpt(transform_object, nodes=nodes)

    # plot inverse transform
    title = f'inverse transform with nodes: \n{nodes}'
    fig, ax = plt.subplots()
    plot.plot_wave(inverse, t_plot, ax, title)
    plt.show()

    # plot initial wave and transform coefficients
    plot.full_plot(final_nodes, data, t_plot, sampling_rate)

### 3.0 Minimum entropy decomposition

Testing the minimum entropy decomposition 

see matlab documentation at https://uk.mathworks.com/help/wavelet/ug/wavelet-packets.html#

In [ ]:
sampling_rate = 25E6

std_noise = 0.01

wavelet = "db1"
levels = 4
mode = "zero"

In [ ]:
import wavelet_transform.utils.training_data as train

frequency = 8E5
pac = -1E5
R_init = 10E-6

min_energy = 0  # Only nodes with energy at least min_energy * total_energy will be considered

data = train.get_data(R_init, frequency, pac, sampling_rate)
data = (data - np.mean(data)) / np.std(data)
data += np.random.normal(0, std_noise, len(data))

plt.plot(data)
plt.show()

In [ ]:
array = np.ones(16)
tr_obj = wpt.dwpt(array, wavelet="haar", mode="zero", min_entropy=True)
print("Entropy of leaf nodes:")
for node in tr_obj.get_leaf_nodes(order="freq"):
    print((node.level, node.node_number), f"{node.entropy:.4f}")

print("\nShannon Entropy of all nodes:")
for level in tr_obj.levels:
    for node in tr_obj.get_level(level):
        print((node.level, node.node_number), f"{node.entropy:.4f}")